This notebook is to calculate SST and precipitation regressions against CANI and EANI during JJA and SON

# Imports

In [1]:
import xarray as xr
import numpy as np
from scipy.stats import t
from dask_jobqueue import PBSCluster
from dask.distributed import Client

# PBSCluster

In [2]:
cluster = PBSCluster(
    cores=1, # The number of cores you want
    memory='16GB', # Amount of memory
    processes=1, # How many processes
    queue='casper', # The type of queue to utilize (/glade/u/apps/dav/opt/usr/bin/execcasper)
    resource_spec='select=1:ncpus=1:mem=16GB', # Specify resources
    account='P93300313', # Input your project ID here
    walltime='04:00:00', # Amount of wall time
    interface='ext', # Interface to use
)
cluster.scale(4)
# Setup your client
client = Client(cluster)

In [3]:
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.232:39343,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [17]:
# client.shutdown()

## Indices

In [4]:
indices = xr.open_dataset("/glade/work/acruz/E3SMv2LE/EANI_CANI_E3SMv2.nc").compute()
indices

<xarray.Dataset> Size: 436kB
Dimensions:  (time: 1212, member: 21)
Coordinates:
    lev      (time) float64 10kB 998.5 998.5 998.5 998.5 ... 998.5 998.5 998.5
    month    (time) int64 10kB 1 2 3 4 5 6 7 8 9 10 ... 3 4 5 6 7 8 9 10 11 12
  * time     (time) object 10kB 1914-01-01 00:00:00 ... 2014-12-01 00:00:00
  * member   (member) int64 168B 0 1 2 3 4 5 6 7 8 ... 13 14 15 16 17 18 19 20
Data variables:
    EANI     (member, time) float64 204kB nan nan 0.7681 ... 1.056 nan nan
    CANI     (member, time) float64 204kB nan nan 0.4377 ... 0.5304 nan nan
Attributes:
    title:        Atlantic Niño Index Timeseries
    description:  Contains EANI, CANI calculated from CESM2.1 LE2
    created:      2025-07-14

## Precipitation Anomaly

In [5]:
PRECCA_ds = xr.open_dataset('/glade/work/acruz/E3SMv2LE/PRECC_anom_hist.nc')
PRECCA_ds = PRECCA_ds.sel(time=slice('1914-01-01', '2014-12-01'))
PRECCA_ds.coords['lon'] = (PRECCA_ds.coords['lon'] + 180) % 360 - 180
PRECCA_ds = PRECCA_ds.sortby(PRECCA_ds.lon)
PRECCA_ds = PRECCA_ds.assign_coords({'member': PRECCA_ds['member']})

In [6]:
PRECCA_ds.compute()

<xarray.Dataset> Size: 6GB
Dimensions:  (lat: 192, time: 1212, member: 21, lon: 288)
Coordinates:
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * time     (time) object 10kB 1914-01-01 00:00:00 ... 2014-12-01 00:00:00
    month    (time) int64 10kB 1 2 3 4 5 6 7 8 9 10 ... 3 4 5 6 7 8 9 10 11 12
  * lon      (lon) float64 2kB -180.0 -178.8 -177.5 -176.2 ... 176.2 177.5 178.8
  * member   (member) int64 168B 0 1 2 3 4 5 6 7 8 ... 13 14 15 16 17 18 19 20
Data variables:
    PRECC    (member, time, lat, lon) float32 6GB -1.866e-11 ... 3.879e-11

# function

In [8]:
def xr_regression(x, y, lag_x=0, lag_y=0, dim="time", alternative="two-sided"):
    """
    From https://stackoverflow.com/questions/52108417/how-to-apply-linear-regression-to-every-pixel-in-a-large-multi-dimensional-array
    requires scipy.stats as t
    Takes two xr.Datarrays of any dimensions (input data could be a 1D
    time series, or for example, have three dimensions e.g. time, lat,
    lon), and returns covariance, correlation, coefficient of
    determination, regression slope, intercept, p-value and standard
    error, and number of valid observations (n) between the two datasets
    along their aligned first dimension.

    Datasets can be provided in any order, but note that the regression
    slope and intercept will be calculated for y with respect to x.

    Inspired by:
    https://hrishichandanpurkar.blogspot.com/2017/09/vectorized-functions-for-correlation.html

    Parameters
    ----------
    x, y : xarray DataArray
        Two xarray DataArrays with any number of dimensions, both
        sharing the same first dimension
    lag_x, lag_y : int, optional
        Optional integers giving lag values to assign to either of the
        data, with lagx shifting x, and lagy shifting y with the
        specified lag amount.
    dim : str, optional
        An optional string giving the name of the dimension on which to
        align (and optionally lag) datasets. The default is 'time'.
    alternative : string, optional
        Defines the alternative hypothesis. Default is 'two-sided'.
        The following options are available:

        * 'two-sided': slope of the regression line is nonzero
        * 'less': slope of the regression line is less than zero
        * 'greater':  slope of the regression line is greater than zero

    Returns
    -------
    regression_ds : xarray.Dataset
        A dataset comparing the two input datasets along their aligned
        dimension, containing variables including covariance, correlation,
        coefficient of determination, regression slope, intercept,
        p-value and standard error, and number of valid observations (n).

    """

    # Shift x and y data if lags are specified
    if lag_x != 0:
        # If x lags y by 1, x must be shifted 1 step backwards. But as
        # the 'zero-th' value is nonexistant, xarray assigns it as
        # invalid (nan). Hence it needs to be dropped
        x = x.shift(**{dim: -lag_x}).dropna(dim=dim)

        # Next re-align the two datasets so that y adjusts to the
        # changed coordinates of x
        x, y = xr.align(x, y)

    if lag_y != 0:
        y = y.shift(**{dim: -lag_y}).dropna(dim=dim)

    # Ensure that the data are properly aligned to each other.
    x, y = xr.align(x, y)

    # Compute data length, mean and standard deviation along dim
    n = y.notnull().sum(dim=dim)
    xmean = x.mean(dim=dim)
    ymean = y.mean(dim=dim)
    xstd = x.std(dim=dim)
    ystd = y.std(dim=dim)

    # Compute covariance, correlation and coefficient of determination
    cov = ((x - xmean) * (y - ymean)).sum(dim=dim) / (n)
    cor = cov / (xstd * ystd)
    r2 = cor**2

    # Compute regression slope and intercept
    slope = cov / (xstd**2)
    intercept = ymean - xmean * slope

    # Compute t-statistics and standard error
    tstats = cor * np.sqrt(n - 2) / np.sqrt(1 - cor**2)
    stderr = slope / tstats

    # Calculate p-values for different alternative hypotheses.
    if alternative == "two-sided":
        pval = t.sf(np.abs(tstats), n - 2) * 2
    elif alternative == "greater":
        pval = t.sf(tstats, n - 2)
    elif alternative == "less":
        pval = t.cdf(np.abs(tstats), n - 2)

    # Wrap p-values into an xr.DataArray
    pval = xr.DataArray(pval, dims=cor.dims, coords=cor.coords)

    # Combine into single dataset
    regression_ds = xr.merge(
        [
            cov.rename("cov").astype(np.float32),
            cor.rename("cor").astype(np.float32),
            r2.rename("r2").astype(np.float32),
            slope.rename("slope").astype(np.float32),
            intercept.rename("intercept").astype(np.float32),
            pval.rename("pvalue").astype(np.float32),
            stderr.rename("stderr").astype(np.float32),
            n.rename("n").astype(np.int16),
        ]
    )

    return regression_ds

# Data selection

In [10]:
# using only the mean of the season as representation of the state that year, regress by year
jja_EANI = indices['EANI'].sel(time=indices['EANI'].time.dt.month.isin([6, 7, 8])).groupby('time.year').mean().compute()
jja_CANI = indices['CANI'].sel(time=indices['CANI'].time.dt.month.isin([6, 7, 8])).groupby('time.year').mean().compute()
jja_EANI

<xarray.DataArray 'EANI' (member: 21, year: 101)> Size: 17kB
array([[ 0.78960837, -0.47745703, -0.1692401 , ...,  1.54193265,
         0.39938639,  0.53710716],
       [ 0.27115393, -0.09261235,  0.86995735, ..., -0.03899549,
         1.36683594,  0.38173927],
       [-0.36649909,  1.13664516,  0.42324534, ...,  0.33118111,
         0.10151934,  0.94092133],
       ...,
       [-0.53606113,  0.22087803, -1.46716821, ..., -0.16725536,
         1.31233453,  0.72498581],
       [-0.09573131, -0.02889102,  0.12712687, ...,  1.06083575,
         0.20636624,  1.43253025],
       [-0.62990181,  0.52536364,  0.9102525 , ...,  0.38904155,
        -0.07127549,  1.42723636]])
Coordinates:
  * member   (member) int64 168B 0 1 2 3 4 5 6 7 8 ... 13 14 15 16 17 18 19 20
  * year     (year) int64 808B 1914 1915 1916 1917 1918 ... 2011 2012 2013 2014

In [11]:
jja_PRECCA = PRECCA_ds['PRECC'].sel(time=PRECCA_ds.time.dt.month.isin([6, 7, 8])).groupby('time.year').mean().compute()
son_PRECCA = PRECCA_ds['PRECC'].sel(time=PRECCA_ds.time.dt.month.isin([9, 10, 11])).groupby('time.year').mean().compute()
jja_PRECCA

<xarray.DataArray 'PRECC' (member: 21, year: 101, lat: 192, lon: 288)> Size: 469MB
array([[[[ 2.65089820e-14,  5.70324021e-14,  5.70324021e-14, ...,
          -4.01443811e-15, -4.01443811e-15, -4.01443811e-15],
         [ 2.65089820e-14,  5.70324021e-14,  5.70324021e-14, ...,
          -4.01443811e-15, -4.01443811e-15, -4.01443811e-15],
         [ 7.24578546e-13,  1.55316677e-12,  1.55170787e-12, ...,
          -1.04201261e-13, -1.04364922e-13, -1.04462879e-13],
         ...,
         [ 5.28699341e-11,  8.58905377e-11,  8.58394744e-11, ...,
           1.97778686e-11,  1.98081742e-11,  1.98263090e-11],
         [ 1.73755177e-11,  3.35232779e-11,  3.35232779e-11, ...,
           1.22775824e-12,  1.22775824e-12,  1.22775824e-12],
         [ 1.73755177e-11,  3.35232779e-11,  3.35232779e-11, ...,
           1.22775824e-12,  1.22775824e-12,  1.22775824e-12]],

        [[-3.81840725e-14, -7.23537102e-14, -7.23537102e-14, ...,
          -4.01443811e-15, -4.01443811e-15, -4.01443811e-15],
         [-3.81840725e-14, -7.23537102e-14, -7.23537102e-14, ...,
          -4.01443811e-15, -4.01443811e-15, -4.01443811e-15],
         [-2.54870344e-13, -4.12385322e-13, -4.12053719e-13, ...,
          -9.69720249e-14, -9.71238809e-14, -9.72147506e-14],
...
          -8.36862871e-12, -8.40532331e-12, -8.42728578e-12],
         [-1.25949977e-12, -1.66129731e-11, -1.66129731e-11, ...,
           1.40939751e-11,  1.40939751e-11,  1.40939751e-11],
         [-1.25949977e-12, -1.66129731e-11, -1.66129731e-11, ...,
           1.40939751e-11,  1.40939751e-11,  1.40939751e-11]],

        [[-3.07976233e-14, -5.24537282e-14, -5.24537282e-14, ...,
          -9.14151584e-15, -9.14151584e-15, -9.14151584e-15],
         [-3.07976233e-14, -5.24537282e-14, -5.24537282e-14, ...,
          -9.14151584e-15, -9.14151584e-15, -9.14151584e-15],
         [-2.54153361e-13, -4.53915036e-13, -4.53523585e-13, ...,
          -5.41291729e-14, -5.42026717e-14, -5.42466496e-14],
         ...,
         [ 5.40127977e-11,  8.31455668e-11,  8.30416916e-11, ...,
           2.48091981e-11,  2.48293747e-11,  2.48414414e-11],
         [-5.45570533e-12, -2.33735618e-11, -2.33735618e-11, ...,
           1.24621520e-11,  1.24621520e-11,  1.24621520e-11],
         [-5.45570533e-12, -2.33735618e-11, -2.33735618e-11, ...,
           1.24621520e-11,  1.24621520e-11,  1.24621520e-11]]]],
      dtype=float32)
Coordinates:
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB -180.0 -178.8 -177.5 -176.2 ... 176.2 177.5 178.8
  * member   (member) int64 168B 0 1 2 3 4 5 6 7 8 ... 13 14 15 16 17 18 19 20
  * year     (year) int64 808B 1914 1915 1916 1917 1918 ... 2011 2012 2013 2014

# Regressions

## Test for 2 members

In [9]:
# x = jja_EANI.sel(member=slice(0, 1))
# y = jja_sst.sel(member=slice(0, 1))

In [10]:
# x

In [11]:
# y

In [12]:
# test = xr_regression(x, y, dim='ntime')
# test

## All members

In [12]:
jja_PRECCAvEANI = xr_regression(jja_EANI, jja_PRECCA, dim='year').compute()
# save results due to long time calculation
jja_PRECCAvEANI.to_netcdf('/glade/work/acruz/E3SMv2LE/Regressions/JJA_PRECCA_EANI_regression.nc')

/glade/u/home/acruz/.local/lib/python3.10/site-packages/xarray/computation/apply_ufunc.py:818: RuntimeWarning: invalid value encountered in sqrt
  result_data = func(*input_data)


In [14]:
jja_PRECCAvCANI = xr_regression(jja_CANI, jja_PRECCA, dim='year').compute()
jja_PRECCAvCANI.to_netcdf('/glade/work/acruz/E3SMv2LE/Regressions/JJA_PRECCA_CANI_regression.nc')

/glade/u/home/acruz/.local/lib/python3.10/site-packages/xarray/computation/apply_ufunc.py:818: RuntimeWarning: invalid value encountered in sqrt
  result_data = func(*input_data)


In [15]:
son_PRECCAvEANI = xr_regression(jja_EANI, son_PRECCA, dim='year')
son_PRECCAvEANI.to_netcdf('/glade/work/acruz/E3SMv2LE/Regressions/SON_PRECCA_EANI_regression.nc')

/glade/u/home/acruz/.local/lib/python3.10/site-packages/xarray/computation/apply_ufunc.py:818: RuntimeWarning: invalid value encountered in sqrt
  result_data = func(*input_data)


In [16]:
son_PRECCAvCANI = xr_regression(jja_CANI, son_PRECCA, dim='year')
son_PRECCAvCANI.to_netcdf('/glade/work/acruz/E3SMv2LE/Regressions/SON_PRECCA_CANI_regression.nc')

/glade/u/home/acruz/.local/lib/python3.10/site-packages/xarray/computation/apply_ufunc.py:818: RuntimeWarning: invalid value encountered in sqrt
  result_data = func(*input_data)
